# import packages

In [2]:
import pandas as pd
import scanpy as sc
import anndata as ad
import os
import warnings
from anndata._core.aligned_df import ImplicitModificationWarning


# set parameters

In [ ]:
input_dir = '../data/raw'
output_dir = '../data/cluster_adata'
os.makedirs(output_dir, exist_ok=True)
normalization = True
log1p = True
PCA = True
features_list = ['name', 'spotId', 'x', 'y']

In [ ]:
# Suppress specific warning from anndata
warnings.filterwarnings("ignore", category=ImplicitModificationWarning)
# Suppress specific FutureWarning from scanpy or general usage
warnings.filterwarnings("ignore", category=FutureWarning)

# preprocess files containing m/z values per pixel

In [ ]:
#read in all files in folder
files = os.listdir(input_dir)

# loop through files and create AnnData object for each file
for file in files:
    df = pd.read_csv(os.path.join(input_dir, file))
    X = df.drop(columns=features_list)
    obs = df[features_list]
    adata = ad.AnnData(X=X, obs=obs)

    if normalization == True:
        sc.pp.normalize_total(adata)

    if log1p == True:
        sc.pp.log1p(adata)

    if PCA == True:
        sc.pp.pca(adata)
        sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca')
    else:
        sc.pp.neighbors(adata, n_neighbors=15)

    sc.tl.leiden(adata, resolution=0.9)
    sc.tl.umap(adata)
    
    adata.write(os.path.join(output_dir, f'{file}.h5ad'))

### filter raw data to contain no background pixels

In [ ]:
#subsetting all_organs csv to only include tissue spots 
raw = pd.read_csv('../data/raw/250311_all_tissues_20um_RMS.csv')
processed = ad.read_h5ad('../data/background_removed/all_organs_annotated_5_1_26.h5ad')


#subset raw by spotId to only include cells in processed 
clean_raw = raw[raw['spotId'].isin(processed.obs['spotId'])].copy()

#save clean_raw to csv
clean_raw.to_csv('../data/background_removed/raw_all_organs_9_1_26.csv', index=False)

### preprocess without normalization as control 

In [ ]:
output_dir = '../data/background_removed'
features_list = ['name', 'spotId', 'x', 'y']

df = pd.read_csv('../data/background_removed/raw_all_organs_9_1_26.csv')
X = df.drop(columns=features_list)
obs = df[features_list]
adata = ad.AnnData(X=X, obs=obs)

#sc.pp.normalize_total(adata)
adata.raw = adata.copy()
sc.pp.log1p(adata)

sc.pp.pca(adata)
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca')


sc.tl.leiden(adata, resolution=0.9)
sc.tl.umap(adata)

adata.write(os.path.join(output_dir, 'non_normalized_all_organs.h5ad'))

# Salivary gland subset

### use processed file to filter raw csv for pixels belonging to salivary glands 


In [ ]:
#subsetting all_organs csv to only include tissue spots from salivary glands
raw = pd.read_csv('../data/raw/250311_all_tissues_20um_RMS.csv')
processed = ad.read_h5ad('../data/background_removed/clean_all_organs_no_border_processed.h5ad')

#subset processed to only include salivary glands
processed = processed[processed.obs['organ'].isin(['RPG', 'SMG', 'SLG', 'parotid']), : ].copy()

#subset raw by spotId to only include cells in processed 
clean_raw = raw[raw['spotId'].isin(processed.obs['spotId'])].copy()

#subset clean_raw to only include mz columns present in processed.var_names
mz_columns = processed.var_names.tolist()
clean_raw = clean_raw[['spotId', 'name', 'x', 'y'] + mz_columns]

#save clean_raw to csv
clean_raw.to_csv('../data/background_removed/clean_salivary_glands.csv', index=False)

### process only salivary glands

In [ ]:
output_dir = '../data/background_removed'

df = pd.read_csv('../data/background_removed/clean_salivary_glands.csv')
X = df.drop(columns=features_list)
obs = df[features_list]
adata = ad.AnnData(X=X, obs=obs)

sc.pp.normalize_total(adata)

sc.pp.log1p(adata)

sc.pp.pca(adata)
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca')


sc.tl.leiden(adata, resolution=0.9)
sc.tl.umap(adata)

adata.write(os.path.join(output_dir, 'clean_salivary_glands_clustered.h5ad'))